# Welcome to the start of your adventure in Agentic AI

### And please do remember to contact me if I can help

And I love to connect: https://www.linkedin.com/in/usamawahabkhan/


### New to Notebooks like this one? Head over to the guides folder!

Just to check you've already added the Python and Jupyter extensions to Cursor, if not already installed:
- Open extensions (View >> extensions)
- Search for python, and when the results show, click on the ms-python one, and Install it if not already installed
- Search for jupyter, and when the results show, click on the Microsoft one, and Install it if not already installed  
Then View >> Explorer to bring back the File Explorer.

And then:
1. Click where it says "Select Kernel" near the top right, and select the option called `.venv (Python 3.12.9)` or similar, which should be the first choice or the most prominent choice. You may need to choose "Python Environments" first.
2. Click in each "cell" below, starting with the cell immediately below this text, and press Shift+Enter to run
3. Enjoy!


In [29]:
# First let's do an import. If you get an Import Error, double check that your Kernel is correct..

from dotenv import load_dotenv
from openai import OpenAI
import os

In [30]:
# Next it's time to load the API keys into environment variables
# If this returns false, see the next cell!

load_dotenv(override=True)

python-dotenv could not parse statement starting at line 7


True

### Wait, did that just output `False`??

If so, the most common reason is that you didn't save your `.env` file after adding the key! Be sure to have saved.

Also, make sure the `.env` file is named precisely `.env` and is in the project root directory (`agents`)

By the way, your `.env` file should have a stop symbol next to it in Cursor on the left, and that's actually a good thing: that's Cursor saying to you, "hey, I realize this is a file filled with secret information, and I'm not going to send it to an external AI to suggest changes, because your keys should not be shown to anyone else."

In [46]:
# Check the key - if you're not using OpenAI, check whichever key you're using! Ollama doesn't need a key.

import os
openai_api_key = os.getenv('DASHSCOPE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")
    


OpenAI API Key exists and begins sk-ws-H.


⭐ 1. Basic Setup (DashScope / Qwen Compatible)

In [ ]:
from openai import OpenAI
import os



In [99]:
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://ws-x76inm0uq4kj1ha6.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
)


In [100]:
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


`messages` is a list of chat turns that tell the model what has happened so far.

- `messages`:
    - a Python list containing message objects
    - each object is a dictionary with:
        - `role`: who sent the message
        - `content`: what the message says

- First entry:
    - `{"role": "system", "content": "You are a helpful assistant."}`
    - This is a system-level instruction
    - It sets the model’s behavior and tone
    - The model uses it as a guiding rule before answering

- Second entry:
    - `{"role": "user", "content": "Add 12 and 30"}`
    - This is the user’s actual prompt
    - It asks the model to perform a task

- Why order matters:
    - messages are processed in sequence
    - the system message appears first so its instruction applies to the whole conversation
    - the user message is next, so the model knows what to respond to

- In practice:
    - the model sees the system instruction: “be helpful”
    - then it sees the user request: “Add 12 and 30”
    - the model generates a response consistent with both messages


    
### 📖 Message Roles

```python
SystemMessage  → Instructions / persona (set by developer)
HumanMessage   → User's input
AIMessage      → Model's previous responses (for memory)
ToolMessage    → Result from a tool call

In [106]:
response = client.chat.completions.create(
    model="qwen3.7-plus",
    max_tokens=500,
    messages=[
        {"role": "user", "content": "Who are you?"}
    ]
)

print(response.choices[0].message.content)


I am Qwen (also known as Tongyi Qianwen), a large language model independently developed by Alibaba Group's Tongyi Lab. I'm here to act as a helpful and empathetic AI thinking partner, ready to assist you with answering questions, writing, coding, brainstorming, or any other tasks you might have. How can I help you today?


⭐ 3. Streaming Response (Tokens as They Arrive)

In [107]:
messages = [{"role": "user", "content": "Who are you"}]

completion = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=messages,
    max_tokens=500,
    stream=True
)

for chunk in completion:
    for chunk in completion:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        if delta.content:
            print(delta.content, end="", flush=True)
    if delta.content:
        print(delta.content, end="", flush=True)


I am Qwen (also known as Tongyi Qianwen), a large language model independently developed by Alibaba Group's Tongyi Lab. How can I help you today?

⭐ 4. Thinking Mode (enable_thinking=True)

In [108]:
completion = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[{"role": "user", "content": "Who are you?"}],
    extra_body={"enable_thinking": True},
    stream=True,
          max_tokens=600
)

is_answering = False
print("\n===== Thinking =====")

for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta

    if getattr(delta, "reasoning_content", None) and not is_answering:
        print(delta.reasoning_content, end="", flush=True)

    if getattr(delta, "content", None):
        if not is_answering:
            print("\n===== Final Answer =====")
            is_answering = True
        print(delta.content, end="", flush=True)



===== Thinking =====

===== Final Answer =====
I am Qwen, a large language model developed by Alibaba Group's Tongyi Lab. I'm here to be a capable and sincerely helpful AI assistant. How can I help you today?

---


### 📖 Concept: How LLMs Work

An LLM is a **next-token predictor** trained on trillions of text tokens. It doesn't "think" — it predicts the statistically most likely continuation of your input.

```
INPUT (tokens):     "The capital of France is"
LLM PREDICTION:     "Paris" (very high probability)
                    "Lyon"  (low probability)
                    "Rome"  (very low probability)
```

### 📖 Key LLM Parameters

| Parameter | Range | Effect |
|-----------|-------|--------|
| **temperature** | 0.0 → 2.0 | 0 = deterministic, 1 = balanced, 2 = chaotic |
| **max_tokens** | 1 → 128k | Max length of response |
| **top_p** | 0.0 → 1.0 | Nucleus sampling (filters unlikely tokens) |
| **presence_penalty** | -2 → 2 | Penalizes repeated topics |
| **frequency_penalty** | -2 → 2 | Penalizes repeated words |

### 📖 Token Fundamentals

```
"Hello, world!"  →  ["Hello", ",", " world", "!"]  →  4 tokens
"Tokenization"   →  ["Token", "ization"]           →  2 tokens
"GPT-4"          →  ["G", "PT", "-", "4"]          →  4 tokens
```

**Rule of thumb:** 1 token ≈ 4 characters ≈ 0.75 words  
**Cost implication:** APIs charge per token — understanding this saves money!

### 📖 LLM Pipeline vs Agentic System

```
LLM PIPELINE (static, one-shot):
  User Input ──► Prompt Template ──► LLM ──► Output
                                              ↑
                                      (that's it, done)

AGENTIC SYSTEM (dynamic, looping):
  User Goal ──► AGENT LOOP ──────────────────────────────────┐
                    │                                         │
                    ├── 1. REASON: "What do I need to do?"   │
                    ├── 2. ACT:    "Call tool X with args Y"  │
                    ├── 3. OBSERVE:"Tool returned Z"          │
                    └── 4. DECIDE: "Goal met? No → loop again"┘
                              "Yes → return final answer"
```


✅ Example with Parameters


In [63]:
response = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user", "content": "Explain artificial intelligence in simple terms."}
    ],
    
    # Sampling & creativity
    temperature=0.7,          # Balanced creativity
    top_p=0.9,                # Consider top 90% probable tokens
    
    # Output length
    max_tokens=150,           # Limit response length
    
    # Repetition control
    presence_penalty=0.6,     # Encourage new ideas
    frequency_penalty=0.3     # Reduce word repetition
)

print(response.choices[0].message.content)


At its most basic level, **Artificial Intelligence (AI) is teaching computers to think, learn, and make decisions somewhat like humans do.**

Instead of giving a computer strict, step-by-step instructions for every single thing it might ever need to do, we give it lots of examples and let it figure out the patterns on its own. 

### A Simple Analogy: Teaching a Child 🐶🐱
Imagine you are teaching a toddler what a dog is. You don’t read them a dictionary definition of a dog’s anatomy. Instead, you point at different dogs and say, "That’s a dog." 

After seeing enough dogs—big ones, small ones, fluffy ones—the toddler’s brain figures out the pattern:


🔒 Deterministic / Precise Output Example


In [65]:
response = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ],
    
    temperature=0.0,          # Fully deterministic
    top_p=1.0,
    max_tokens=10,
    presence_penalty=0,
    frequency_penalty=0
)

print(response.choices[0].message.content)

2 + 2 = 4.


⚙️ Quick Parameter Guide


temperature (0–2)

0.0 → predictable, exact answers
0.7 → natural balance ✅
1.5+ → creative, diverse



top_p (0–1)

Controls probability mass (alternative to temperature)
Lower = safer answers, Higher = more diverse



max_tokens

Limits response size (important for cost + control)



presence_penalty (-2 → 2)

Higher = model introduces new topics



frequency_penalty (-2 → 2)

Higher = reduces repeated words/phrases

# Token Counting & Cost Estimation (Fixed & Clean)


In [66]:
# ── Cell 1.1 — Token counting & cost estimation ──────────────────
import tiktoken

# Encoding (works well for GPT-4 / Qwen-style models)
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> dict:
    tokens = enc.encode(text)
    return {
        "text": text[:80] + "..." if len(text) > 80 else text,
        "token_count": len(tokens),
        "char_count": len(text),
        "chars_per_token": round(len(text) / len(tokens), 2) if tokens else 0,
        "estimated_cost_per_1M_input": "$0.50"  # approximate
    }

# ── Examples ──────────────────────────────────────────────────────
examples = [
    "Hello, world!",
    "Tokenization is the process of converting text into smaller units.",
    "The quick brown fox jumps over the lazy dog.",
    "LangChain is a framework for developing applications powered by large language models.",
    "قرآن کریم",   # Arabic text
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
]

print(f"{'Text (truncated)':<60} {'Tokens':>7} {'Chars':>6} {'C/T':>5}")
print("-" * 80)

for text in examples:
    result = count_tokens(text)
    print(f"{result['text']:<60} {result['token_count']:>7} {result['char_count']:>6} {result['chars_per_token']:>5}")

print()
print("💡 Key insight: Non-English text uses MORE tokens per character")
print("💡 Code is efficient — keywords compress well")

Text (truncated)                                              Tokens  Chars   C/T
--------------------------------------------------------------------------------
Hello, world!                                                      4     13  3.25
Tokenization is the process of converting text into smaller units.      12     66   5.5
The quick brown fox jumps over the lazy dog.                      10     44   4.4
LangChain is a framework for developing applications powered by large language m...      14     86  6.14
قرآن کریم                                                          9      9   1.0
def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)      23     73  3.17

💡 Key insight: Non-English text uses MORE tokens per character
💡 Code is efficient — keywords compress well


🔗 Cell 1.2 — Combine with Your client (Real Usage)
This shows how to estimate tokens BEFORE sending a request 

In [68]:
# ── Cell 1.2 — Token-aware request ────────────────────────────────

def count_message_tokens(messages):
    total = 0
    for msg in messages:
        total += len(enc.encode(msg["content"]))
    return total

messages = [
    {"role": "user", "content": "Explain artificial intelligence in simple terms."}
]

input_tokens = count_message_tokens(messages)

print(f"🧮 Estimated input tokens: {input_tokens}")

response = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=messages,
    max_tokens=120,
    temperature=0.7
)

output_text = response.choices[0].message.content
output_tokens = len(enc.encode(output_text))

print(f"🧾 Output tokens: {output_tokens}")
print(f"💰 Total tokens: {input_tokens + output_tokens}")
print()
print(output_text)

🧮 Estimated input tokens: 8
🧾 Output tokens: 119
💰 Total tokens: 127

Think of Artificial Intelligence (AI) as **teaching a computer to learn and make decisions, much like a human does.**

Instead of a human programmer typing in exact, step-by-step instructions for every single situation, we give the computer a lot of information and let it figure out the rules on its own. 

Here is the easiest way to understand it, broken down into three parts:

### 1. The Analogy: Teaching a Child
Imagine you want to teach a toddler what a "cat" is. You don't give them a scientific definition of a feline. Instead


## 💸 Quick Cost Estimation Helper


In [69]:
def estimate_cost(tokens, price_per_1M=0.50):
    return round((tokens / 1_000_000) * price_per_1M, 6)

total_tokens = input_tokens + output_tokens
cost = estimate_cost(total_tokens)

print(f"💵 Estimated cost: ${cost}")

💵 Estimated cost: $6.3e-05


 Pro Tips (Advanced)
✅ 1. Track token usage live
Wrap your API calls:

In [70]:
def chat_with_tracking(messages):
    input_tokens = count_message_tokens(messages)
    
    response = client.chat.completions.create(
        model="qwen3.7-plus",
        messages=messages
    )
    
    output = response.choices[0].message.content
    output_tokens = len(enc.encode(output))
    
    total = input_tokens + output_tokens
    
    print(f"Input: {input_tokens} | Output: {output_tokens} | Total: {total}")
    
    return output


✅ 2. Token budget control


In [71]:
MAX_BUDGET = 500

if input_tokens > MAX_BUDGET:
    print("⚠️ Prompt too long — trim it first!")

---
## 🧪 Section 5 — Prompt Engineering: CoT, Few-shot, ToT (25 min)

### 📖 Concept: Why Prompting Matters

The same model can give **completely different quality answers** depending on how you ask the question. Prompt engineering is the skill of crafting inputs that reliably produce high-quality outputs.

### 📖 Prompting Techniques

#### 1️⃣ Zero-Shot Prompting
```
Classify this review as positive/negative: "The food was amazing!"
→ Positive  (model guesses from training)
```

#### 2️⃣ Few-Shot Prompting
```
Review: "Great food!" → Positive
Review: "Terrible service." → Negative
Review: "Okay experience, nothing special." → ?
→ Neutral  (model learns pattern from examples)
```

#### 3️⃣ Chain-of-Thought (CoT)
```
WITHOUT CoT:  "Is 17 × 23 = 391?" → "Yes" (often wrong)
WITH CoT:     "Think step by step: 17 × 23 = ?"
              → "17 × 20 = 340, 17 × 3 = 51, 340 + 51 = 391. Yes."
```

#### 4️⃣ Tree-of-Thought (ToT)
```
Generate 3 different approaches to the problem.
Evaluate each approach.
Select the best and elaborate.
→ Forces exploration of solution space before committing
```

### 📖 When to Use Each

| Technique | Best For |
|-----------|---------|
| Zero-shot | Simple, well-defined tasks |
| Few-shot | Classification, format-following |
| CoT | Math, logic, multi-step reasoning |
| ToT | Complex problems, creative tasks |
| ReAct | Tasks requiring tool use |

In [72]:
# ============================================================
# 1️⃣ ZERO‑SHOT — simple tasks
# ============================================================
response_zero_shot = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user",
         "content": "Classify this review as positive or negative: 'The food was amazing!'"}
    ],
    temperature=0.0,
    max_tokens=20
)
print("Zero-shot:", response_zero_shot.choices[0].message.content)


# ============================================================
# 2️⃣ FEW‑SHOT — pattern learning
# ============================================================
response_few_shot = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user",
         "content":
         "Review: 'Great food!' → Positive\n"
         "Review: 'Terrible service.' → Negative\n"
         "Review: 'Okay experience, nothing special.' → ?"}
    ],
    temperature=0.0,
    max_tokens=20
)
print("Few-shot:", response_few_shot.choices[0].message.content)


# ============================================================
# 3️⃣ CHAIN‑OF‑THOUGHT (CoT) — step-by-step reasoning
# ============================================================
response_cot = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user",
         "content": "Think step by step: Is 17 × 23 = 391?"}
    ],
    temperature=0.0,
    max_tokens=100
)
print("CoT:", response_cot.choices[0].message.content)


# ============================================================
# 4️⃣ TREE‑OF‑THOUGHT (ToT) — multiple reasoning paths
# ============================================================
response_tot = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user",
         "content":
         "Use Tree-of-Thought: Generate 3 solution paths, evaluate them, "
         "and choose the best.\nProblem: How can a company allocate limited parking spaces fairly?"}
    ],
    temperature=0.2,
    max_tokens=200
)
print("ToT:", response_tot.choices[0].message.content)


# ============================================================
# 5️⃣ ReAct — reasoning + tool use
# ============================================================
response_react = client.chat.completions.create(
    model="qwen3.7-plus",
    temperature=0.0,
    max_tokens=300,
    messages=[
        {"role": "system",
         "content":
         "Use ReAct format:\n"
         "Thought: reasoning\n"
         "Action: tool name\n"
         "Action Input: JSON\n"
         "Observation: tool result\n"
         "Final Answer: final response\n"
         "Tools: search(query), calculator(expression)"},

        {"role": "user",
         "content": "What is the population of Dubai and what is 17 × 23?"},

        {"role": "assistant",
         "content":
         "Thought: I should search for Dubai's population.\n"
         "Action: search\n"
         "Action Input: {\"query\": \"Dubai population 2024\"}"},

        {"role": "tool", "content": "Dubai population is approximately 3.65 million."},

        {"role": "assistant",
         "content":
         "Thought: Now calculate 17 × 23.\n"
         "Action: calculator\n"
         "Action Input: {\"expression\": \"17 * 23\"}"},

        {"role": "tool", "content": "391"},

        {"role": "assistant",
         "content":
         "Thought: I have both results.\n"
         "Final Answer: Dubai's population is about 3.65 million and 17 × 23 = 391."}
    ]
)
print("ReAct:", response_react.choices[0].message.content)


Zero-shot: Positive
Few-shot: Neutral
CoT: Yes, 17 × 23 = 391. Here is the step-by-step verification using two different methods:

**Method 1: Breaking it down (Distributive Property)**
You can break 23 into 20 + 3 to make the multiplication easier:
1. Multiply 17 by 20: 17 × 20 = 340
2. Multiply 17 by 3: 17 × 
ToT: Here is a Tree-of-Thought (ToT) analysis to solve the problem of fairly allocating limited company parking spaces. 

---

### **Problem Definition**
**Goal:** Allocate limited parking spaces fairly among employees.
**Constraints:** Limited supply, diverse employee needs (commute distance, health, income, family status), and the need to maintain morale and perceived organizational justice.

---

### **Step 1: Generate 3 Solution Paths**

#### **Path 1: Needs-Based & Hierarchical Allocation**
* **Concept:** Allocate spaces based on objective, verifiable needs and business criticality. 
* **Mechanism:** 
  1. Legally mandated accommodations first (e.g., ADA/disability requirem

## Introduction to Function Calling

Function calling lets the model invoke Python functions during a chat completion. Instead of just returning text, the assistant can:
- decide when a tool is needed,
- provide structured arguments,
- and let your code execute the function.

This is useful for:
- calculations,
- accessing external APIs,
- retrieving data,
- and building agentic workflows.

Typical flow:
1. define a function,
2. expose it as a tool,
3. let the model choose a tool,
4. execute the tool in Python,
5. return the result back to the model for the final response.

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    # If the environment variable is not set, replace it with your Model Studio API key: api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://ws-x76inm0uq4kj1ha6.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
)

messages = [{"role": "user", "content": "Who are you"}]
completion = client.chat.completions.create(
    model="qwen3.7-plus",  # You can replace this with another deep thinking models
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)
is_answering = False  # Indicates whether the response phase has started
print("\n" + "=" * 20 + "Thinking process" + "=" * 20)
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "Full response" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)

In [50]:
def add_numbers(a: int, b: int):
    return {"result": a + b}


Step 2 — Describe it to the model

In [51]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two integers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    }
]


Step 4 — Detect tool call

In [52]:
response = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[{"role": "user", "content": "Add 12 and 30"}],
    tools=tools,
    tool_choice="auto"
)


Step 5 — Execute the function

In [53]:
result = add_numbers(**args)
print(f"Tool response: {result}")

Tool response: {'result': 42}


Step 6 — Send result back to model

In [54]:
name = tool_call.function.name
followup = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[
        {"role": "user", "content": "Add 12 and 30"},
        msg,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": name,
            "content": json.dumps(result)
        }
    ]
)

print(followup.choices[0].message.content)


12 + 30 = 42


⭐ 6. JSON Mode (Structured Output)

In [55]:
response = client.chat.completions.create(
    model="qwen3.7-plus",
    messages=[{"role": "user", "content": "Give me a JSON with name and age"}],
    response_format={"type": "json_object"}
)

data = json.loads(response.choices[0].message.content)
print(data)


{'name': 'John Doe', 'age': 30}


⭐ 7. Error Handling (Best Practice)

In [56]:
try:
    response = client.chat.completions.create(
        model="qwen3.7-plus",
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:
    print("API Error:", e)


# 🦙 Ollama Integration Guide (Python)

## ✅ 1.

In [64]:
import requests
import json

url = "http://localhost:11434/api/chat"

payload = {
    "model": "qwen3:1.7b",
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
    "stream": False
}

response = requests.post(url, json=payload)
print(response.json()["message"]["content"])


I'm an AI assistant designed to help with various tasks and provide information. I can answer questions, offer support, and assist with tasks like writing, problem-solving, and more. My goal is to be helpful and friendly. Feel free to ask me anything you'd like to know! 😊


In [80]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"   # required but ignored
)


✅ 3. Basic Chat Completion (Qwen3‑1.7B)

In [81]:
response = client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[
        {"role": "user", "content": "Who are you?"}
    ]
)

print(response.choices[0].message.content)


Hello! I'm an AI assistant designed to help you with information, answer questions, and assist in various tasks. I'm here to provide accurate, helpful, and respectful responses to your queries. While I don't have consciousness or emotions, I'm programmed to be empathetic and courteous in my interactions. Feel free to ask me anything, and I'll do my best to assist you! 😊


✅ 4. Streaming Response

In [ ]:
stream = client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[{"role": "user", "content": "Explain AI in simple words."}],
    stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)


Artificial Intelligence (AI) is like a smart helper that can do things on its own, without being told exactly what to do. It learns from experience and helps machines make decisions or solve problems. For example, AI can recognize images, understand speech, or even play games. It's like a brain in a computer, but instead of thinking with neurons, it uses data and patterns to "learn" and do tasks. AI is used in things like smartphones, self-driving cars, and medical tools to make life easier and more efficient. 🧠✨

# 🦙 Using `qwen3:1.7b` with Ollama (LangChain + OpenAI Client)

`qwen3:1.7b` is a lightweight local model — fast, efficient, and great for low-resource setups ✅

---

## 🚀 1. Run the Model in Ollama

```bash
ollama pull qwen3:1.7b
ollama run qwen3:1.7b

In [82]:
from langchain_community.llms import Ollama

llm = Ollama(model="qwen3:1.7b")

response = llm.invoke("Who are you?")
print(response)

I am an AI assistant developed by Alibaba Group. My name is Qwen, and I am designed to help you with your questions, provide information, and assist with various tasks. I can answer questions, offer guidance, and support with tasks like language translation, problem-solving, and more. I aim to be helpful, friendly, and efficient in my interactions. While I don't have personal experiences or emotions, I am here to support you in any way I can! Let me know how I can assist you. 😊


⚙️ With Parameters


In [83]:
llm = Ollama(
    model="qwen3:1.7b",
    temperature=0.7,
    top_p=0.9
)

print(llm.invoke("Explain AI simply."))


Artificial Intelligence (AI) is like a smart robot that can learn and make decisions on its own. It uses data and patterns to solve problems, recognize things, or do tasks that humans might do. For example, a virtual assistant like Siri or Alexa can understand what you say, learn your preferences, and help you with tasks. AI doesn’t think like humans, but it can do things like recognize images, play games, or translate languages. It’s all about making machines "learn" from the information they’re given and use that knowledge to improve over time. 🤖✨


## 5. Function Calling (Tool Calling) With Ollama

In [84]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two integers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    }
]


In [85]:
response = client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[{"role": "user", "content": "Add 12 and 30"}],
    tools=tools,
    tool_choice="auto"
)


In [86]:
msg = response.choices[0].message

if msg.tool_calls:
    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)


In [87]:
def add_numbers(a, b):
    return {"result": a + b}

result = add_numbers(**args)


In [88]:
followup = client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[
        {"role": "user", "content": "Add 12 and 30"},
        msg,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": "add_numbers",
            "content": json.dumps(result)
        }
    ]
)

print(followup.choices[0].message.content)


The result of adding 12 and 30 is $\boxed{42}$.


🧠 6. Memory (Simple Agent Memory)

In [89]:
memory = [
    {"role": "system", "content": "You are a helpful agent."}
]

def agent(user_input):
    memory.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=memory
    )
    answer = response.choices[0].message.content
    memory.append({"role": "assistant", "content": answer})
    return answer

print(agent("Hello"))
print(agent("What did I just say?"))


Hello! How can I assist you today?
You said **"Hello"** first, and then asked a question. Here's the full conversation flow:

1. You: "Hello"  
2. Assistant: "Hello! How can I assist you today?"  
3. You: "What did I just say?"  

Let me know if you'd like to continue the conversation or need anything else! 😊


🧠 7. Planning (Qwen Thinking Mode)

In [90]:
extra_body={"enable_thinking": True}


In [91]:
response = client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[{"role": "user", "content": "Solve 12 * 17"}],
    extra_body={"enable_thinking": True}
)


In [92]:
from IPython.display import Markdown, display

display(Markdown(response.choices[0].message.content))

To solve the expression:

$$
12 \times 17
$$

---

We perform the multiplication step-by-step:

$$
12 \times 17 = (10 + 2) \times 17 = 10 \times 17 + 2 \times 17 = 170 + 34 = 204
$$

---

### Final Answer:

$$
\boxed{204}
$$

---

**Note:** The original problem was written as $12 \times 17 / \ldots$, which leaves the denominator unspecified. If the denominator were, for example, 1, the result would still be 204. However, without additional context, we cannot determine the exact value of the denominator. Thus, the most reasonable assumption is that the expression is simply $12 \times 17$, and its value is:

$$
\boxed{204}
$$

# 🧠 Multi‑Tool Agent Flow (LangGraph‑Style)

               ┌──────────────────────────┐
               │        User Input        │
               └─────────────┬────────────┘
                             ▼
               ┌──────────────────────────┐
               │   Add message to memory  │
               └─────────────┬────────────┘
                             ▼
               ┌──────────────────────────┐
               │  Model Call (Planning)   │
               │  - Reads memory          │
               │  - Decides next action   │
               │  - May request a tool    │
               └─────────────┬────────────┘
                     Yes ┌───┴───────┐ No
                         ▼           ▼
        ┌────────────────────────┐   ┌────────────────────────┐
        │   Tool Call Detected   │   │   Normal Model Reply   │
        └─────────────┬──────────┘   └─────────────┬──────────┘
                      ▼                            ▼
        ┌────────────────────────┐      ┌────────────────────────┐
        │ Execute Python Tool    │      │  Add reply to memory   │
        │ (add, multiply, etc.)  │      └─────────────┬──────────┘
        └─────────────┬──────────┘                    ▼
                      ▼                    ┌────────────────────────┐
        ┌────────────────────────┐         │      Return Answer     │
        │  Send Tool Result Back │         └────────────────────────┘
        │     to the Model       │
        └─────────────┬──────────┘
                      ▼
        ┌────────────────────────┐
        │ Final Model Response   │
        │ (uses tool result)     │
        └─────────────┬──────────┘
                      ▼
        ┌────────────────────────┐
        │ Add final answer to    │
        │ memory (for recall)    │
        └─────────────┬──────────┘
                      ▼
        ┌────────────────────────┐
        │     Return Answer      │
        └────────────────────────┘


In [93]:
from openai import OpenAI
import json

# -----------------------------------------
# 1. Connect to Ollama (Local)
# -----------------------------------------
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"   # required but ignored
)

# -----------------------------------------
# 2. Agent Memory
# -----------------------------------------
memory = [
    {"role": "system", "content": "You are a local offline agent. Use tools when needed."}
]

# -----------------------------------------
# 3. Define Multiple Tools
# -----------------------------------------

def add_numbers(a: int, b: int):
    return {"result": a + b}

def multiply_numbers(a: int, b: int):
    return {"result": a * b}

def get_weather(city: str):
    return {"weather": f"Sunny in {city} (offline mock)"}

# -----------------------------------------
# 4. Expose Tools to the Model
# -----------------------------------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two integers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "multiply_numbers",
            "description": "Multiply two integers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get offline weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    }
]

# -----------------------------------------
# 5. Agent Step (LangGraph-style loop)
# -----------------------------------------
def agent(user_input):
    memory.append({"role": "user", "content": user_input})

    # First model call (planning + tool selection)
    response = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=memory,
        tools=tools,
        tool_choice="auto",
        extra_body={"enable_thinking": True}
    )

    msg = response.choices[0].message
    memory.append(msg)

    # If the model wants to call a tool
    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        # Execute the correct tool
        if name == "add_numbers":
            result = add_numbers(**args)
        elif name == "multiply_numbers":
            result = multiply_numbers(**args)
        elif name == "get_weather":
            result = get_weather(**args)
        else:
            result = {"error": "Unknown tool"}

        # Return tool result to model
        tool_msg = {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": name,
            "content": json.dumps(result)
        }
        memory.append(tool_msg)

        # Final model response
        final = client.chat.completions.create(
            model="qwen3:1.7b",
            messages=memory
        )
        answer = final.choices[0].message.content
        memory.append({"role": "assistant", "content": answer})
        return answer

    # No tool call → normal answer
    answer = msg.content
    memory.append({"role": "assistant", "content": answer})
    return answer

# -----------------------------------------
# 6. Test the Multi-Tool Agent
# -----------------------------------------
print(agent("Hello, who are you?"))
print(agent("Add 12 and 30"))
print(agent("Multiply 7 and 8"))
print(agent("What is the weather in Dubai?"))
print(agent("What did I ask earlier?"))


I'm an AI assistant designed to help with tasks like adding and multiplying numbers, and providing weather information. Let me know how I can assist you! 😊
The result of adding 12 and 30 is **42**.
The result of multiplying 7 and 8 is **56**.
The weather in Dubai is **Sunny**.
You asked the following questions earlier:
1. "Hello, who are you?"  
2. "Add 12 and 30"  
3. "Multiply 7 and 8"  
4. "What is the weather in Dubai?"  

Let me know if you'd like to explore further! 😊


In [57]:
%pip install langgraph openai


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from openai import OpenAI
import json

# -----------------------------------------
# 1. Local LLM (Ollama)
# -----------------------------------------
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# -----------------------------------------
# 2. Agent State
# -----------------------------------------
class AgentState(dict):
    messages: list

# -----------------------------------------
# 3. Tools
# -----------------------------------------
def add_numbers(a: int, b: int):
    return {"result": a + b}

def multiply_numbers(a: int, b: int):
    return {"result": a * b}

def get_weather(city: str):
    return {"weather": f"Sunny in {city} (offline mock)"}

tools = {
    "add_numbers": add_numbers,
    "multiply_numbers": multiply_numbers,
    "get_weather": get_weather
}

# -----------------------------------------
# 4. LLM Node (Qwen3‑1.7B)
# -----------------------------------------
def call_llm(state: AgentState):
    response = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=state["messages"],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": name,
                    "description": f"Tool: {name}",
                    "parameters": {"type": "object", "properties": {}}
                }
            }
            for name in tools.keys()
        ],
        tool_choice="auto",
        extra_body={"enable_thinking": True}
    )

    msg = response.choices[0].message
    return {"messages": state["messages"] + [msg]}

# -----------------------------------------
# 5. Tool Node
# -----------------------------------------
def tool_router(state: AgentState):
    msg = state["messages"][-1]

    if not msg.tool_calls:
        return END

    tool_call = msg.tool_calls[0]
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = tools[name](**args)

    tool_msg = {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "name": name,
        "content": json.dumps(result)
    }

    return {"messages": state["messages"] + [tool_msg]}

# -----------------------------------------
# 6. Build Graph
# -----------------------------------------
graph = StateGraph(AgentState)

graph.add_node("llm", call_llm)
graph.add_node("tools", tool_router)

graph.set_entry_point("llm")
graph.add_edge("llm", "tools")
graph.add_edge("tools", "llm")

app = graph.compile()

# -----------------------------------------
# 7. Run Local Agent
# -----------------------------------------
def agent(user_input):
    state = {"messages": [{"role": "user", "content": user_input}]}
    result = app.invoke(state)
    return result["messages"][-1]["content"]

print(agent("Hello, who are you?"))
print(agent("Add 12 and 30"))
print(agent("Multiply 7 and 8"))
print(agent("What is the weather in Dubai?"))


In [97]:
from langgraph.graph import StateGraph, END
from openai import OpenAI
import json

# -----------------------------------------
# 1. Local LLM (Ollama)
# -----------------------------------------
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# -----------------------------------------
# 2. Agent State
# -----------------------------------------
class AgentState(dict):
    messages: list

# -----------------------------------------
# 3. Tools
# -----------------------------------------
def add_numbers(a: int, b: int):
    return {"result": a + b}

def multiply_numbers(a: int, b: int):
    return {"result": a * b}

def get_weather(city: str):
    return {"weather": f"Sunny in {city} (offline mock)"}

tools_map = {
    "add_numbers": add_numbers,
    "multiply_numbers": multiply_numbers,
    "get_weather": get_weather
}

# ✅ Tool schema (IMPORTANT)
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "multiply_numbers",
            "description": "Multiply two numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    }
]

# -----------------------------------------
# 4. LLM Node
# -----------------------------------------
def call_llm(state: AgentState):
    response = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=state["messages"],
        tools=tools_schema,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    return {
        "messages": state["messages"] + [msg]
    }

# -----------------------------------------
# 5. Tool Node
# -----------------------------------------
def tool_router(state: AgentState):
    msg = state["messages"][-1]

    if not msg.tool_calls:
        return {"messages": state["messages"]}

    tool_call = msg.tool_calls[0]
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = tools_map[name](**args)

    tool_msg = {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "name": name,
        "content": json.dumps(result)
    }

    return {
        "messages": state["messages"] + [tool_msg]
    }

# -----------------------------------------
# 6. Control Flow
# -----------------------------------------
def should_continue(state: AgentState):
    msg = state["messages"][-1]

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        return "tools"

    return END

# -----------------------------------------
# 7. Build Graph
# -----------------------------------------
graph = StateGraph(AgentState)

graph.add_node("llm", call_llm)
graph.add_node("tools", tool_router)

graph.set_entry_point("llm")

graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

graph.add_edge("tools", "llm")

app = graph.compile()

# -----------------------------------------
# 8. Agent Runner
# -----------------------------------------
def agent(user_input):
    state = {
        "messages": [
            {
                "role": "system",
                "content": "You must use tools for math and weather. Be concise."
            },
            {"role": "user", "content": user_input}
        ]
    }

    result = app.invoke(state)

    last_msg = result["messages"][-1]

    # ✅ FIX: handle object properly
    if hasattr(last_msg, "content"):
        return last_msg.content
    else:
        return last_msg["content"]

# -----------------------------------------
# ✅ Test
# -----------------------------------------
print(agent("Hello, who are you?"))
print(agent("Add 12 and 30"))
print(agent("Multiply 7 and 8"))
print(agent("What is the weather in Dubai?"))

I'm an AI assistant designed to help with math problems, weather inquiries, and other tasks. Let me know how I can assist you! 🌡️🧮
The result of adding 12 and 30 is **42**.
The result of multiplying 7 and 8 is **56**.
The weather in Dubai is currently sunny (offline mock).


# Congratulations!

That was a small, simple step in the direction of Agentic AI, with your new environment!

Next time things get more interesting...

<table style="margin: 0; text-align: left; width:100%">
    <tr>
              <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>
        </td>
    </tr>
</table>

In [ ]:
# First create the messages:

messages = [{"role": "user", "content": "Something here"}]

# Then make the first call:

response =

# Then read the business idea:

business_idea = response.

# And repeat! In the next message, include the business idea within the message